# 17 OCR feedback to planner and execution bridge

Promote rows that became canonical-ready after OCR-augmented suggestions back into planner-style outputs. Optionally merge them with the latest plan and rebuild execution manifests.


In [ ]:
from pathlib import Path
import sys
from datetime import datetime
import pandas as pd

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('OUTPUT_DIR =', OUTPUT_DIR)


In [ ]:
from src.ocr_feedback_to_execution import (
    OCRPromotionConfig,
    build_ocr_promoted_plan,
    merge_ocr_into_plan,
    ocr_promotion_summary,
)

try:
    from src.executor import build_execution_bundle, save_manifest_bundle, manifest_summary
    HAS_EXECUTOR = True
except Exception as exc:
    HAS_EXECUTOR = False
    EXECUTOR_IMPORT_ERROR = repr(exc)

HAS_EXECUTOR


In [ ]:
def latest_output(prefix: str, suffix: str = '.parquet'):
    candidates = sorted(OUTPUT_DIR.glob(f'{prefix}_*{suffix}'), key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0] if candidates else None

OCR_FEEDBACK_RERUN_PATH = latest_output('ocr_canonical_feedback_rerun')
EXISTING_PLAN_PATH = latest_output('plan_with_feedback') or latest_output('plan_dry_run')

print('OCR_FEEDBACK_RERUN_PATH =', OCR_FEEDBACK_RERUN_PATH)
print('EXISTING_PLAN_PATH =', EXISTING_PLAN_PATH)
if not OCR_FEEDBACK_RERUN_PATH:
    raise FileNotFoundError('No ocr_canonical_feedback_rerun_*.parquet found in data/outputs')


In [ ]:
PROMOTE_ONLY_NEWLY_READY = True
MERGE_WITH_EXISTING_PLAN = True
BUILD_EXECUTION_MANIFESTS = True
DEFAULT_FULL_ROOT = None

config = OCRPromotionConfig(
    promote_only_newly_ready=PROMOTE_ONLY_NEWLY_READY,
    default_full_root=DEFAULT_FULL_ROOT,
)
config


In [ ]:
ocr_feedback_rerun = pd.read_parquet(OCR_FEEDBACK_RERUN_PATH)
existing_plan = pd.read_parquet(EXISTING_PLAN_PATH) if (MERGE_WITH_EXISTING_PLAN and EXISTING_PLAN_PATH) else pd.DataFrame()

promoted = build_ocr_promoted_plan(ocr_feedback_rerun, config=config)
merged_plan = merge_ocr_into_plan(existing_plan, promoted)

print('promoted summary:', ocr_promotion_summary(promoted))
print('merged rows:', len(merged_plan))


In [ ]:
def _show(df, cols, n=20):
    safe_cols = [c for c in cols if c in df.columns]
    display(df[safe_cols].head(n))

_show(promoted, ['relative_path', 'planner_action', 'planner_reason', 'planner_target_relative_path', 'accepted_fields', 'became_canonical_ready', 'promotion_source'])
display(promoted['planner_action'].fillna('missing').value_counts().rename_axis('planner_action').reset_index(name='count'))


In [ ]:
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
promoted_csv = OUTPUT_DIR / f'plan_promoted_from_ocr_feedback_{stamp}.csv'
promoted_parquet = OUTPUT_DIR / f'plan_promoted_from_ocr_feedback_{stamp}.parquet'
merged_csv = OUTPUT_DIR / f'plan_with_ocr_feedback_{stamp}.csv'
merged_parquet = OUTPUT_DIR / f'plan_with_ocr_feedback_{stamp}.parquet'

promoted.to_csv(promoted_csv, index=False, encoding='utf-8-sig')
promoted.to_parquet(promoted_parquet, index=False)
merged_plan.to_csv(merged_csv, index=False, encoding='utf-8-sig')
merged_plan.to_parquet(merged_parquet, index=False)

print(promoted_csv)
print(promoted_parquet)
print(merged_csv)
print(merged_parquet)


In [ ]:
if BUILD_EXECUTION_MANIFESTS:
    if not HAS_EXECUTOR:
        raise ImportError(f'src.executor could not be imported: {EXECUTOR_IMPORT_ERROR}')
    bundle = build_execution_bundle(merged_plan)
    stem = f'ocr_feedback_bridge_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
    paths = save_manifest_bundle(bundle, OUTPUT_DIR, stem=stem)
    print('manifest summary:', manifest_summary(bundle))
    for name, path in paths.items():
        print(name, '->', path)
else:
    print('BUILD_EXECUTION_MANIFESTS = False; skipped manifest build')
